# 3. Embed & Load

 Purpose  : Embed chunks_to_embed.jsonl using AWS Bedrock
            Titan V2 and upsert into Qdrant vector store.


# Qdrant client & AWS bedrock Test

In [1]:
from dotenv import load_dotenv
from qdrant_client import QdrantClient
import os
import boto3

load_dotenv(override=True)
print(" Environment variables loaded from .env file")

qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"), 
    timeout=60,
)

print(" QdrantClient initialized successfully")

session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION", "us-east-1")
    ) 

bedrock_client = session.client("bedrock-runtime")

print(" boto3 Session and Bedrock client created")

 Environment variables loaded from .env file
 QdrantClient initialized successfully
 boto3 Session and Bedrock client created


In [2]:
def run_Qdrant_AWS_test():
    """Tests if both vector DB and AWS connections are working."""
    
    # Test Qdrant
    collections = qdrant_client.get_collections()
    print(f"Qdrant: Connected | Collections found: {len(collections.collections)}")
    
    # Test AWS (STS = Security Token Service)
    sts_client = session.client("sts")
    identity = sts_client.get_caller_identity()

    print(f" AWS   : Connected")
    print(f"   Account ID : {identity['Account']}")
    print(f"   User ARN   : {identity['Arn']}")

# Run the test
run_Qdrant_AWS_test()

Qdrant: Connected | Collections found: 3
 AWS   : Connected
   Account ID : 462397577625
   User ARN   : arn:aws:iam::462397577625:user/llmops-developer


# Validation Function

In [3]:
def validate_chunk(chunk):
    """
    Returns True if the chunk has required structure:
    - 'id' field
    - 'text_content' field
    - 'payload' dictionary containing 'metadata'
    Otherwise returns False.
    """
    if not isinstance(chunk, dict):
        return False
    
    if 'id' not in chunk or 'text_content' not in chunk:
        return False
    
    if 'payload' not in chunk or not isinstance(chunk.get('payload'), dict):
        return False
    
    if 'metadata' not in chunk['payload']:
        return False
    
    return True

# Batch Generator using yield

In [4]:
import json

def stream_batches(file_path, batch_size, log_errors=False):
    """
    Generator function that reads a JSONL file line by line.
    This approach avoids loading the entire file into memory.
    """
    batch = []
    invalid_count = 0
    
    with open(file_path, 'r', encoding='utf-8') as file:
        for line_num, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
                
            try:
                chunk = json.loads(line)
                if validate_chunk(chunk):
                    batch.append(chunk)
                    
                    if len(batch) >= batch_size:
                        yield batch
                        batch = []
                else:
                    invalid_count += 1
            except json.JSONDecodeError:
                invalid_count += 1
                if log_errors:
                    print(f"JSON decode error on line {line_num}")
                continue
            except Exception as e:
                invalid_count += 1
                if log_errors:
                    print(f"Error on line {line_num}: {type(e).__name__}")
                continue
    
    if batch:
        yield batch
    
    if invalid_count > 0:
        print(f"Skipped {invalid_count} invalid or malformed records.")

In [5]:
file_path = "../data/chunks/chunks_to_embed.jsonl"

batch_generator = stream_batches(file_path=file_path, batch_size=2, log_errors=True)
first_batch = next(batch_generator)

print("First batch received:")
for chunk in first_batch:
    print(chunk)

First batch received:
{'id': '500180_INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json_chunk_pnl_quarterly_interest_income', 'text_content': 'HDFC BANK LIMITED - Quarterly Interest Income (Core Banking Revenue):\n- InterestEarned: 8,706,694.00\n- InterestOrDiscountOnAdvancesOrBills: 6,411,401.00\n- InterestOnBalancesWithReserveBankOfIndiaAndOtherInterBankFunds: 77,353.00\n- OtherInterest: 159,929.00', 'payload': {'metadata': {'company_name': 'HDFC BANK LIMITED', 'scrip_code': '500180', 'industry': 'Banking - Private Sector', 'currency': 'INR Crores', 'report_type': 'Consolidated', 'fiscal_year_end': 'March', 'fiscal_year': 'FY2026', 'quarter': 'Q3', 'period_type': 'Quarterly', 'chunk_name': 'chunk_pnl_quarterly_interest_income', 'source': 'INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB_parsed.json'}, 'raw_data': {'InterestEarned': {'value': 8706694.0, 'unit': 'INR Crores'}, 'InterestOrDiscountOnAdvancesOrBills': {'value': 6411401.0, 'unit': 'INR Crores'}, 'Interest

#  Embedding Function

In [6]:
import json
import boto3
from botocore.exceptions import ClientError

def get_bedrock_embedding(text, client, model_id):
    """
    Returns embedding vector for a single text using Amazon Titan Embeddings V2.
    """
    if not text or not isinstance(text, str):
        raise ValueError("Input text must be a non-empty string")
    
    request_body = {
        "inputText": text,
        "dimensions": 1024,
        "normalize": True
    }
    
    try:
        response = client.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(request_body)
        )
        
        response_body = json.loads(response.get('body').read())
        embedding = response_body.get('embedding')
        
        if embedding is None:
            raise ValueError("No embedding returned from Bedrock")
            
        return embedding
        
    except Exception as e:
        raise RuntimeError(f"Failed to get embedding: {str(e)}")

In [7]:
import time

def embed_batch(texts, client, model_id):
    """
    Embeds a list of texts and returns a list of vectors.
    """
    embeddings = []
    
    for text in texts:
        success = False
        last_exception = None
        
        for attempt in range(3):
            try:
                vector = get_bedrock_embedding(text, client, model_id)
                embeddings.append(vector)
                success = True
                break
            except ClientError as e:
                last_exception = e
                error_code = e.response['Error']['Code']
                if error_code == 'ThrottlingException' or 'TooManyRequests' in str(e):
                    time.sleep(1)  # Wait one second before retrying
                    continue
                else:
                    raise  # Re-raise if it's not a throttling error
            except Exception as e:
                last_exception = e
                time.sleep(1)
                continue
        
        if not success:
            print(f"Warning: Failed to embed text after 3 attempts. Error: {last_exception}")
            embeddings.append([0.0] * 1024)
    
    return embeddings

In [8]:
# Test the embedding functions
sample_texts = [
    "This is a test sentence for embedding.",
    "RAG pipelines require high quality vector representations.",
    "AWS Bedrock Titan V2 produces 1024 dimensional embeddings."
]

model_id = "amazon.titan-embed-text-v2:0"

vectors = embed_batch(sample_texts, bedrock_client, model_id)

print(f"Successfully generated {len(vectors)} embeddings")
print(f"Dimension of first vector: {len(vectors[0])}")

# Verify all vectors have correct dimension
for i, vector in enumerate(vectors):
    assert len(vector) == 1024, f"Vector {i} has incorrect dimension"
    print(f"Vector {i} dimension verified: {len(vector)}")
    
print("All embeddings generated successfully with dimension 1024.")

Successfully generated 3 embeddings
Dimension of first vector: 1024
Vector 0 dimension verified: 1024
Vector 1 dimension verified: 1024
Vector 2 dimension verified: 1024
All embeddings generated successfully with dimension 1024.


# Log Failed Texts for Reprocessing

In [9]:
import json
from datetime import datetime

def log_failed_texts(texts, log_file="failed_texts.jsonl", reason="embedding_failed"):
    """Logs failed texts to a JSONL file for later reprocessing."""
    timestamp = datetime.utcnow().isoformat()
    with open(log_file, 'a', encoding='utf-8') as f:
        for text in texts:
            record = {
                "text": text,
                "timestamp": timestamp,
                "reason": reason,
                "model_id": "amazon.titan-embed-text-v2:0"
            }
            f.write(json.dumps(record) + '\n')
    print(f"Logged {len(texts)} failed texts to {log_file}")

# Single Text Embedding Function

In [10]:
def get_bedrock_embedding(text, client, model_id):
    """Single text embedding - used as fallback."""
    if not text or not isinstance(text, str):
        raise ValueError("Input text must be a non-empty string")
    
    request_body = {
        "inputText": text,
        "dimensions": 1024,
        "normalize": True
    }
    
    response = client.invoke_model(
        modelId=model_id,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response.get('body').read())
    embedding = response_body.get('embedding')
    
    if embedding is None:
        raise ValueError("No embedding returned from Bedrock")
        
    return embedding

#  Robust Batch Embedding (Batch + Single Fallback)

In [11]:
import json
import time
from botocore.exceptions import ClientError

def embed_batch_robust(texts, client, model_id, batch_size=5, max_retries=3, log_file="failed_texts.jsonl"):
    """
    Safely embeds texts using Titan V2.
    """
    if not texts:
        return []
    
    all_embeddings = []
    failed_texts = []
    
    # Process texts in small batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_success = False
        
        # Try batch API first (list input)
        if len(batch) > 1:
            for attempt in range(max_retries):
                try:

                    request_body  = {
                            "inputText": batch,          
                            "dimensions": 1024,
                            "normalize": True
                        }

                    response = client.invoke_model(
                        modelId=model_id,
                        contentType="application/json",
                        accept="application/json",
                        body=json.dumps(request_body) 
                    )
                    response_body = json.loads(response.get('body').read())
                    embeddings = response_body.get('embedding')
                    
                    if isinstance(embeddings, list) and len(embeddings) == len(batch):
                        all_embeddings.extend(embeddings)
                        batch_success = True
                        break
                except Exception:
                    time.sleep(1)
                    continue
        
        # Fallback: Embed one by one with retry (Most reliable for Titan V2)
        if not batch_success:
            for text in batch:
                success = False
                for attempt in range(max_retries):
                    try:
                        vector = get_bedrock_embedding(text, client, model_id)
                        all_embeddings.append(vector)
                        success = True
                        break
                    except (ClientError, Exception) as e:
                        if attempt == max_retries - 1:
                            failed_texts.append(text)
                        else:
                            time.sleep(1)
        
    # Log any texts that completely failed
    if failed_texts:
        log_failed_texts(failed_texts, log_file=log_file, reason="embedding_failed_after_retries")
    
    return all_embeddings

In [12]:
sample_texts = [
    "This is a test sentence for embedding.",
    "RAG pipelines require high quality vector representations.",
    "Titan V2 works best with single string inputText.",
    "Failed texts are logged for later reprocessing."
]

model_id = "amazon.titan-embed-text-v2:0"

vectors = embed_batch_robust(
    texts=sample_texts,
    client=bedrock_client,
    model_id=model_id,
    batch_size=5,           
    max_retries=3,
    log_file="failed_texts.jsonl"
)

print(f"Generated {len(vectors)} embeddings")
print(f"Embedding dimension: {len(vectors[0]) if vectors else 0}")

Generated 4 embeddings
Embedding dimension: 1024


# The Vector Store Blueprint

In [13]:
# Check if the collection 'nifty50_financials' already exists
collections = qdrant_client.get_collections()
collection_names = [collection.name for collection in collections.collections]

collection_name = "nifty50_financials"
collection_exists = collection_name in collection_names

print(f"Collection '{collection_name}' exists: {collection_exists}")

Collection 'nifty50_financials' exists: True


In [14]:
from qdrant_client.http.models import Distance, VectorParams

# Create collection if it doesn't exist
if not collection_exists:
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=1024,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists. Skipping creation.")

Collection 'nifty50_financials' already exists. Skipping creation.


In [15]:
from qdrant_client.http.models import PayloadSchemaType

def sync_payload_indexes(client, collection_name, desired_fields):
    """
    Synchronizes payload indexes with desired fields.
    - Checks current indexes.
    - Adds missing indexes.
    - Removes unwanted indexes.
    - Includes rollback on failure to maintain consistent state.
    """
    # Get current payload schema state
    current_info = client.get_collection(collection_name).payload_schema
    current_fields = set(current_info.keys())
    desired_fields = set(desired_fields)
    
    # Identify differences
    to_add = desired_fields - current_fields
    to_remove = current_fields - desired_fields
    
    applied_changes = []
    
    try:
        # Process removals first
        for field in to_remove:
            client.delete_payload_index(collection_name, field)
            applied_changes.append(('remove', field))
            print(f"Removed index for field: {field}")
        
        # Process additions
        for field in to_add:
            client.create_payload_index(
                collection_name=collection_name,
                field_name=field,
                field_schema=PayloadSchemaType.KEYWORD
            )
            applied_changes.append(('add', field))
            print(f"Created index for field: {field}")
            
    except Exception as e:
        print(f"Failure during index sync: {e}. Rolling back changes...")
        for action, field in reversed(applied_changes):
            try:
                if action == 'add':
                    client.delete_payload_index(collection_name, field)
                else:
                    client.create_payload_index(
                        collection_name=collection_name,
                        field_name=field,
                        field_schema=PayloadSchemaType.KEYWORD
                    )
            except Exception as rollback_error:
                print(f"Rollback failed for {field}: {rollback_error}")
        print("Restored to previous index state.")
        return False
    
    print(f"Payload index synchronization completed for collection '{collection_name}'.")
    return True


# Usage for our RAG pipeline
desired_fields = ['company_name', 'fiscal_year', 'quarter']
sync_payload_indexes(qdrant_client, collection_name, desired_fields)

Payload index synchronization completed for collection 'nifty50_financials'.


True

In [16]:
# Verify collection configuration and status
collection_info = qdrant_client.get_collection(collection_name)

print("\n=== Collection Verification ===")
print(f"Vector Dimension : {collection_info.config.params.vectors.size}")
print(f"Distance Metric  : {collection_info.config.params.vectors.distance}")
print(f"Points Count     : {collection_info.points_count}")
print(f"Indexed Fields   : {list(collection_info.payload_schema.keys())}")
print("\nCollection setup completed successfully!")


=== Collection Verification ===
Vector Dimension : 1024
Distance Metric  : Cosine
Points Count     : 0
Indexed Fields   : ['fiscal_year', 'quarter', 'company_name']

Collection setup completed successfully!


# The Integration Loop

In [17]:
import hashlib
import uuid
from qdrant_client.http.models import PointStruct

def build_qdrant_point(chunk, vector):
    """
    Creates a PointStruct for Qdrant.
    Uses a deterministic UUID based on chunk['id'] to ensure the same
    content always gets the same point ID.
    """
    # Create deterministic UUID from chunk id
    chunk_id_str = str(chunk.get('id', ''))
    md5_hash = hashlib.md5(chunk_id_str.encode('utf-8')).hexdigest()
    point_id = uuid.UUID(md5_hash)          # Return UUID object (recommended)
    
    # Build payload - required fields + flattened metadata
    metadata = chunk.get('payload', {}).get('metadata', {})
    
    payload = {
        'id': chunk.get('id'),
        'text_content': chunk.get('text_content')
    }
    payload.update(metadata)
    
    return PointStruct(
        id=point_id,          # UUID object is accepted and serialized correctly
        vector=vector,
        payload=payload
    )

In [18]:
# Main processing loop - updated to work with fixed point ID logic
file_path = "../data/chunks/chunks_to_embed.jsonl"
batch_size = 8
collection_name = "nifty50_financials"
model_id = "amazon.titan-embed-text-v2:0"

batch_generator = stream_batches(file_path, batch_size)

total_processed = 0
batch_count = 0

for batch in batch_generator:
    batch_count += 1
    
    texts = [chunk['text_content'] for chunk in batch]
    vectors = embed_batch_robust(texts, bedrock_client, model_id)
    
    points = []
    for chunk, vector in zip(batch, vectors):
        point = build_qdrant_point(chunk, vector)
        points.append(point)
    
    qdrant_client.upsert(
        collection_name=collection_name,
        points=points,
        wait=True                     # Ensures upsert completes before next batch
    )
    
    total_processed += len(batch)
    
    if batch_count % 5 == 0:
        print(f"Processed {total_processed} chunks so far.")
        
print(f"Completed. Total chunks upserted: {total_processed}")

Processed 36 chunks so far.
Completed. Total chunks upserted: 36


# Verification 

In [34]:
from typing import List
from qdrant_client.http.models import ScoredPoint

def semantic_search(query, client, collection_name, bedrock_client, model_id, limit=3):
    """
    Performs pure vector similarity search.
    1. Embeds the query using the existing get_bedrock_embedding function.
    2. Searches the collection for the top 'limit' most similar vectors.
    3. Returns list of ScoredPoint objects containing score and payload.
    """
    # Convert query text to embedding
    query_vector = get_bedrock_embedding(query, bedrock_client, model_id)
    
    # Perform vector search
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=limit,
        with_payload=True
    )
    
    return search_result.points

In [35]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Example of Filtered Semantic Search (Vector search + Payload filter)
def filtered_search(query, client, collection_name, bedrock_client, model_id, limit=3):
    """
    Combines vector similarity with payload filtering.
    Only returns results where company_name exactly matches the filter.
    """
    query_vector = get_bedrock_embedding(query, bedrock_client, model_id)
    
    # Define filter for exact match on metadata field
    company_filter = Filter(
        must=[
            FieldCondition(
                key="company_name",
                match=MatchValue(value="HDFC Bank")
            )
        ]
    )
    
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=company_filter,
        limit=limit,
        with_payload=True
    )
    
    return search_result.points

In [37]:
# Smoke test - run a sample query and display results
test_query = "What was the quarterly revenue performance of major banks?"

results = semantic_search(
    query=test_query,
    client=qdrant_client,
    collection_name="nifty50_financials",
    bedrock_client=bedrock_client,
    model_id="amazon.titan-embed-text-v2:0",
    limit=3
)

print("Semantic Search Results:\n")
print(f"{'Score':<8} | {'Company':<15} | Text Preview")
print("-" * 70)

for point in results:
    score = round(point.score, 4)
    company = point.payload.get('company_name', 'N/A')
    text_preview = point.payload.get('text_content', '')[:200] + "..."
    print(f"{score:<8} | {company:<15} | {text_preview}")

Semantic Search Results:

Score    | Company         | Text Preview
----------------------------------------------------------------------
0.2265   | HDFC BANK LIMITED | HDFC BANK LIMITED - Segment Revenue (Quarterly + Annual combined view):
- SegmentRevenue: 54,987,302.00
- SegmentRevenueFromOperations: 37,854,287.00
- InterSegmentRevenue: 17,133,015.00...
0.2037   | HDFC BANK LIMITED | HDFC BANK LIMITED - Quarterly Interest Income (Core Banking Revenue):
- InterestEarned: 8,706,694.00
- InterestOrDiscountOnAdvancesOrBills: 6,411,401.00
- InterestOnBalancesWithReserveBankOfIndiaAndOt...
0.1998   | HDFC BANK LIMITED | HDFC BANK LIMITED - Quarterly Non-Interest Income:
- RevenueOnInvestments: 2,058,011.00
- OtherIncome: 3,986,033.00
- Income: 12,692,727.00...
